# Synthetic Floor 7-Stage — GPU Blender Renderer (Colab)

Production-grade renderer for progressive construction stages on a Colab GPU.

## Key features
- **Frame-level resume** — each frame saved immediately; disconnect loses only 1 frame.
- **Drive persistence** — outputs sync every ~60 s + at stage boundaries.
- **Parallel rendering** — `WORKERS` concurrent Blender processes (strided frames).
- **HDR lighting + AO** — Filmic tone-mapping + ambient occlusion (fixes blown-out look).
- **Hyper-realistic camera** — looks around, turns 180°, crouches/rises, full coverage.
- **Choose example + start stage** — no forced restart.
- **`.blend` export** — download and open in Blender on Windows/macOS.
- **Valid IFC4** per stage for the main pipeline.

## Quick start
1. Set `CONFIG`, `START_STAGE`, `WORKERS` in cell 1.
2. Run cells 1–4 (setup + render).
3. If disconnected: re-run 1–4 with the same `RUN_NAME` → resumes.

| `CONFIG` | Building | Floor |
|---|---|---|
| `config/scene.yaml` | single room | raw slab |
| `config/scene_office.yaml` | office | tile |
| `config/scene_loft.yaml` | loft studio | wood |
| `config/scene_warehouse.yaml` | warehouse | epoxy |

Set runtime to **GPU**: *Runtime → Change runtime type → T4/A100*.


## 1. Settings — choose example, start stage, workers


In [ ]:
# ====== EDIT THESE ======
RUN_NAME    = 'demo'                       # same name = RESUME
CONFIG      = 'config/scene_office.yaml'   # which example
START_STAGE = 1                            # 1..7
WORKERS     = 1                            # parallel Blender procs (1 on T4; 2+ on A100)
PRESET      = 'balanced'                   # debug | balanced | hq
FRAMES      = 120                          # per stage (120=4s; 900=30s)
SAVE_BLEND  = True                         # downloadable .blend
STRICT      = True                         # fail on blank frames
# ========================


## 2. GPU, Drive, clone


In [ ]:
import os, sys, subprocess
try:
    print('GPU:', subprocess.check_output(['nvidia-smi','-L']).decode().strip())
except Exception:
    print('No GPU! Runtime -> Change runtime type -> T4/A100')
DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = f'/content/drive/MyDrive/MyCon_Colab/synthetic_floor_7stage/{RUN_NAME}'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive:', DRIVE_ROOT)
except Exception as e:
    print('Drive N/A:', e)
if not os.path.isdir('/content/MyCon'):
    subprocess.check_call(['git','clone','--depth','1','https://github.com/sayyedalimrj/MyCon.git','/content/MyCon'])
else:
    subprocess.call(['git','-C','/content/MyCon','pull','--ff-only'])
os.chdir('/content/MyCon/repository')
EXAMPLE='examples/synthetic_floor_7stage'
sys.path.insert(0, EXAMPLE+'/src')
print('Ready:', os.getcwd())


## 3. Install Blender + deps


In [ ]:
%%bash
set -e
if [ ! -f /content/blender/blender ]; then
  wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/b.tar.xz
  mkdir -p /content/blender && tar -xf /tmp/b.tar.xz -C /content/blender --strip-components=1 && rm /tmp/b.tar.xz
fi
apt-get -qq install -y libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libgl1 libegl1 libxfixes3 >/dev/null 2>&1||true
/content/blender/blender --version 2>&1|head -2


In [ ]:
!pip install -q numpy Pillow PyYAML trimesh imageio imageio-ffmpeg ifcopenshell scipy


## 4. Render (resumable, parallel, HDR/AO, starts at START_STAGE)

Re-run after a disconnect: finished stages + already-rendered frames are skipped.
Each worker renders a disjoint frame subset (no locking). HDR lighting + AO from config.


In [ ]:
parts = [
    f'PYTHONPATH={EXAMPLE}/src python3 {EXAMPLE}/scripts/run_blender_gpu.py',
    f'--config {EXAMPLE}/{CONFIG}',
    '--blender /content/blender/blender',
    f'--preset {PRESET} --frames {FRAMES} --device OPTIX',
    f'--start-stage {START_STAGE}',
    f'--workers {WORKERS}',
    '--mount-drive' if DRIVE_ROOT else '',
    f'--drive-root "{DRIVE_ROOT}"' if DRIVE_ROOT else '',
    '--save-blend' if SAVE_BLEND else '',
    '--resume',
    '--strict-render' if STRICT else '',
]
cmd = ' '.join(p for p in parts if p)
print('Running:', cmd)
os.system(cmd)


## 5. Live progress + preview


In [ ]:
import json
from pathlib import Path
from synthetic_floor.scene_spec import load_scene_spec
SPEC = load_scene_spec(Path(EXAMPLE)/CONFIG)
RENDER_DIR = SPEC.output.root / 'blender_renders'
for sd in sorted(RENDER_DIR.glob('stage_*')):
    pf = sd/'render_progress.json'
    if pf.exists():
        d=json.loads(pf.read_text())
        print(f"{sd.name}: {d.get('frames_done')}/{d.get('frames_total')} ({d.get('percent')}%) status={d.get('status')}")
    n=len(list((sd/'rgb').glob('frame_*.png'))) if (sd/'rgb').exists() else 0
    print(f'   frames on disk: {n}')


In [ ]:
import numpy as np
from PIL import Image
from IPython.display import display
stage_dirs=sorted(RENDER_DIR.glob('stage_*'))
rgb_dir=(stage_dirs[-1]/'rgb') if stage_dirs else RENDER_DIR
pngs=sorted(rgb_dir.glob('frame_*.png'))
print(f'{len(pngs)} frame(s)')
if pngs:
    mid=pngs[len(pngs)//2]
    arr=np.asarray(Image.open(mid).convert('RGB'),dtype=np.float32)
    lum=arr.mean(axis=2); w=float((lum>250).mean())
    print(f'{mid.name}: mean={lum.mean():.1f} std={lum.std():.2f} white={w:.3f}')
    print('OK' if (w<0.97 and lum.std()>3) else 'LOOKS BLANK')
    display(Image.open(mid))


## 6. Resume status + IFC + downloads


In [ ]:
import json
OUTPUT=SPEC.output.root
rs=OUTPUT/'manifests'/'run_state_blender_gpu.json'
if rs.exists():
    b=json.loads(rs.read_text())
    print('run_id:',b.get('run_id'),'| done:',b.get('in_progress')==False)
    for sid,st in sorted(b.get('stages',{}).items()):
        dm=st.get('done_marker') or {}
        print(f"  stage {sid}: complete={st['complete']} frames={dm.get('rgb_frame_count')}")
ifc=OUTPUT/'bim'/'stage_07.ifc'
if ifc.exists():
    try:
        import ifcopenshell; m=ifcopenshell.open(str(ifc))
        print('IFC:',{c:len(m.by_type(c)) for c in ('IfcColumn','IfcBeam','IfcWall','IfcWindow','IfcDoor')})
    except: print('IFC present')


In [ ]:
from IPython.display import FileLink, display
print('Videos:')
for f in sorted((OUTPUT/'video').glob('*.mp4')):
    print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f}MB)'); display(FileLink(str(f)))
print('\nBlend zips:')
bd=OUTPUT/'blend'
if bd.exists():
    for f in sorted(bd.glob('*.zip')):
        print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f}MB)'); display(FileLink(str(f)))


## 7. Reference: how it all works

**Resume:** re-run cells 1–4 with the same `RUN_NAME`. Done stages skipped; partial stages continue frame-by-frame.

**Stage selection:** `START_STAGE=5` renders 5→7 only. Use `--stage-only X` for exactly one stage. It does NOT restart from the beginning.

**Example selection:** set `CONFIG`. Each has its own output folder.

**Parallel:** `WORKERS` Blender processes per stage (strided frames). Keep 1 on T4; 2+ on A100.

**Drive:** outputs mirror every ~60 s. Cross-system pulls use content-hash delta (0 bytes for unchanged files).

**HDR lighting & AO:** low ambient sky, sun key-light, Filmic + contrast, −0.2 EV, AO contact shadows. All from config.

**Another device:** copy the run folder to the new Drive, same `RUN_NAME`+`CONFIG`, run.
